In [2]:
import os
import re
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import requests
from io import StringIO
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import icepython as icexl
from eventregistry import *
from jinja2 import Environment, FileSystemLoader

# ==========================================
# 1. FETCH DATA & CREATE FIG 1 (Summer/Winter NBP)
# ==========================================
print("Fetching Fig 1 data...")
current_date = datetime.date.today()
ed = current_date.strftime('%Y-%m-%d')
sd = (current_date - timedelta(days=365)).strftime('%Y-%m-%d')

fields = ['volume', 'close']
my_data = icexl.get_timeseries(['GWMS 1!-ICE'], fields, granularity='D', start_date=sd, end_date=ed)
my_data2 = icexl.get_timeseries(['GWMS 2!-ICE'], fields, granularity='D', start_date=sd, end_date=ed)

dates, vols, values = zip(*my_data[1:])  
dates2, vols2, values2 = zip(*my_data2[1:])  
formatted_dates = [date for date in dates]

num_points = len(formatted_dates)
indices = [0, num_points//3, 2*num_points//3, num_points-1]
selected_dates = [formatted_dates[i] for i in indices]

fig1 = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.1, row_heights=[0.99, 0.01])
fig1.add_trace(go.Scatter(x=formatted_dates, y=values, mode='lines+markers',marker=dict(color='deepskyblue'), name='Summer 26'),row=1, col=1)
fig1.add_trace(go.Scatter(x=formatted_dates, y=values2, mode='lines+markers',marker=dict(color='darkviolet'), name='Winter 26'),row=1, col=1)

fig1.update_layout(
    title={'text': 'Summer and Winter 26 Daily NBP Prices 12 Month Lag', 'x': 0.5, 'xanchor': 'center'},
    legend=dict(x=0.01, y=0.98, bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1),
    yaxis_title='p/therm',
    xaxis=dict(tickmode='array', tickvals=[formatted_dates[i] for i in indices], ticktext=selected_dates),
    font=dict(size=24), 
    autosize=True, margin=dict(l=50, r=30, t=60, b=50)
)

# ==========================================
# 2. FETCH DATA & CREATE FIG 2 (Forward NBP WoW)
# ==========================================
print("Fetching Fig 2 data...")
sd_7 = (current_date - timedelta(days=7)).strftime('%Y-%m-%d')
lom = ['GWM ' + str(i) + '!-ICE' for i in range(1,13)]
ed = (current_date - timedelta(days=1)).strftime('%Y-%m-%d')
sd = (current_date - timedelta(days=8)).strftime('%Y-%m-%d')

my_data_wow = icexl.get_timeseries(lom, ['Last'], granularity='D', start_date=sd, end_date=ed)

last_week_p = my_data_wow[1][1:]

wow_pull = icexl.get_quotes(lom, ['ICE Theoretical Price'])
current_theo_p = [i[1] for i in wow_pull[1:]]

def get_next_12_months():
    start_d = datetime.date.today() + datetime.timedelta(days=1)
    months_list = []
    curr_d = start_d
    for _ in range(12):
        curr_m = curr_d.month
        curr_y = curr_d.year
        next_m = curr_m + 1
        next_y = curr_y
        if next_m > 12:
            next_m = 1
            next_y += 1
        curr_d = datetime.date(next_y, next_m, 1)
        months_list.append(curr_d.strftime('%b'))
    return months_list

labels = get_next_12_months()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=labels, y=last_week_p, mode='lines+markers', name='Last Week', marker=dict(color='deepskyblue'), line=dict(color='deepskyblue')))
fig2.add_trace(go.Scatter(x=labels, y=current_theo_p, mode='lines+markers', name='Current Price', marker=dict(color='darkviolet'), line=dict(color='darkviolet')))

fig2.update_layout(
    title={'text': 'Forward NBP Curve Change Week on Week', 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Month', yaxis_title='P/th',
    legend=dict(x=0.01, y=0.98, bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1),
    hovermode='x unified',
    font=dict(size=24), autosize=True, margin=dict(l=50, r=30, t=60, b=50)
)

# ==========================================
# 3. FETCH DATA & CREATE FIG 3 (Wind & Temp)
# ==========================================
print("Fetching Fig 3 data (Weather/Wind)...")
urls = {'Temp':"https://app.enappsys.com/datadownload?code=isem/weather/tempair/national/history&currency=EUR&delimiter=comma&minavmax=false&pass=211225245243225231229237225238177178183161&res=hh&tag=csv&timezone=WET&user=diarmuid.egan@flogas.ie", 
        'Wind':'https://app.enappsys.com/datadownload?code=isem/elec/renewables/wind/onshore/forecast/history&currency=EUR&delimiter=comma&minavmax=false&pass=211225245243225231229237225238177178183161&res=hh&tag=csv&timezone=WET&user=diarmuid.egan@flogas.ie'}

def gen_url(base):
    start_d = current_date.strftime('%Y%m%d0000')
    end_d = (current_date + timedelta(days=0)).strftime('%Y%m%d2330')
    return f"{urls[base]}&start={start_d}&end={end_d}"

def motel_data(base):
    resp = requests.get(gen_url(base))
    if resp.status_code != 200:
        raise Exception(f"Failed to fetch data: {resp.status_code}")
    data = StringIO(resp.text)
    df = pd.read_csv(data)
    df['Date (WET)'] = df['Date (WET)'].str.strip('[]')
    df['Date (WET)'] = pd.to_datetime(df['Date (WET)'], format='%d/%m/%Y %H:%M')
    df = df.drop(index=0).reset_index(drop=True)
    if base == 'Temp':
        df['LATEST_TEMP'] = df['LATEST'].astype(float)
        df['Date'] = df['Date (WET)']
        return df[['Date','LATEST_TEMP']]
    if base == 'Wind':
        df['LATEST_WIND'] = df['LATEST FORECAST (EnAppSys)'].astype(float)
        df['Date'] = df['Date (WET)']
        return df[['Date','LATEST_WIND']]

Wind = motel_data('Wind')
Temp = motel_data('Temp')

temp_summary = Temp.groupby(Temp['Date'].dt.date).agg(min_temp=('LATEST_TEMP', 'min'), max_temp=('LATEST_TEMP', 'max')).reset_index()
temp_summary['Date'] = pd.to_datetime(temp_summary['Date']) + pd.Timedelta(hours=12)

fig3 = make_subplots(specs=[[{"secondary_y": True}]])
fig3.add_trace(go.Scatter(x=Wind['Date'], y=Wind['LATEST_WIND'], name='Wind', mode='lines', line=dict(color='darkviolet')), secondary_y=False)
fig3.add_trace(go.Bar(x=temp_summary['Date'], y=temp_summary['max_temp'] - temp_summary['min_temp'], base=temp_summary['min_temp'], name='Temp Range', marker=dict(color='deepskyblue', opacity=0.6), width=1000 * 3600 * 4), secondary_y=True)

for i, row in temp_summary.iterrows():
    fig3.add_annotation(x=row['Date'], y=row['max_temp'], yref="y2", text=f"{row['max_temp']:.1f}", showarrow=False, font=dict(color="black", size=18), yshift=10)
    fig3.add_annotation(x=row['Date'], y=row['min_temp'], yref="y2", text=f"{row['min_temp']:.1f}", showarrow=False, font=dict(color="black", size=18), yshift=-10)

fig3.update_layout(
    title={'text': 'Wind and Temperature Forecast', 'x': 0.5, 'xanchor': 'center'}, 
    showlegend=False, font=dict(size=24), autosize=True, margin=dict(l=50, r=50, t=60, b=50),
    yaxis=dict(showgrid=False), yaxis2=dict(showgrid=False)
)
fig3.update_yaxes(title_text="Wind (MW)", secondary_y=False, range=[0, 5000])
fig3.update_yaxes(title_text="Temperature (°C)", secondary_y=True)

# ==========================================
# 4. FETCH DATA & CREATE FIG 4 (GAS Forward Curve WoW)
# ==========================================
print("Fetching Fig 4 data (GAS Curve)...")
ed_gas = (current_date - timedelta(days=0)).strftime('%Y-%m-%d')
sd_gas = (current_date - timedelta(days=7)).strftime('%Y-%m-%d')
lom_gas = ['GAS ' + str(i) + '!-ICE' for i in range(1,13)]

my_data_gas = icexl.get_timeseries(lom_gas, ['Last'], granularity='D', start_date=sd_gas, end_date=sd_gas)
last_week_gas_p = my_data_gas[1][1:]

wow_pull_gas = icexl.get_quotes(lom_gas, ['ICE Theoretical Price'])
current_theo_gas_p = [i[1] for i in wow_pull_gas[1:]]

fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=labels, # Reusing labels array initialized in Fig 2
    y=last_week_gas_p, 
    mode='lines+markers', 
    name='Last Week',
    marker=dict(color='deepskyblue'),
    line=dict(color='deepskyblue')
))
fig4.add_trace(go.Scatter(
    x=labels, 
    y=current_theo_gas_p, 
    mode='lines+markers', 
    name='Current Price',
    marker=dict(color='darkviolet'),
    line=dict(color='darkviolet')
))

fig4.update_layout(
    title={'text': 'Forward Diesel Curve Change Week on Week', 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Month', yaxis_title='Price',
    legend=dict(x=0.01, y=0.98, bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1),
    hovermode='x unified',
    font=dict(size=24), autosize=True, margin=dict(l=50, r=30, t=60, b=50)
)

# ==========================================
# 5. FETCH NEWS (EventRegistry)
# ==========================================
print("Fetching News...")
er = EventRegistry(apiKey = '1312ee84-a9f9-4fc6-95a4-0ed13b135ec3')
nat_gas_concept_uri = er.getConceptUri("Natural Gas")
q = QueryArticles(conceptUri=nat_gas_concept_uri, lang="eng")
q.setRequestedResult(RequestArticlesInfo(sortBy="date", count=100))
results = er.execQuery(q)

unique_list = []
seen_titles = set()
if results.get('articles') and results['articles'].get('results'):
    for article in results['articles']['results']:
        title = article['title'].strip()
        source_name = article['source']['title']
        if re.search('gas', title, re.IGNORECASE) and title not in seen_titles:
            unique_list.append(f"{title} - {source_name}")
            seen_titles.add(title)

display_news = unique_list + unique_list 

# ==========================================
# 6. GENERATE HTML DASHBOARD
# ==========================================
print("Generating Final HTML Dashboard...")

fig1_html = fig1.to_html(full_html=False, include_plotlyjs='cdn')
fig2_html = fig2.to_html(full_html=False, include_plotlyjs=False)
fig3_html = fig3.to_html(full_html=False, include_plotlyjs=False)
fig4_html = fig4.to_html(full_html=False, include_plotlyjs=False)

env = Environment(loader=FileSystemLoader('.'))
template = env.get_template('tv_template.html')

html_content = template.render(
    fig1_html=fig1_html,
    fig2_html=fig2_html,
    fig3_html=fig3_html,
    fig4_html=fig4_html,
    headlines=display_news
)

output_file = 'index.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Success! {output_file} has been generated.")

Fetching Fig 1 data...
Fetching Fig 2 data...
Fetching Fig 3 data (Weather/Wind)...
Fetching Fig 4 data (GAS Curve)...
Fetching News...
Generating Final HTML Dashboard...
Success! index.html has been generated.


In [3]:
import os
import re
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import requests
from io import StringIO
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import icepython as icexl
from eventregistry import *
from jinja2 import Environment, FileSystemLoader

# ==========================================
# 1. FETCH DATA & CREATE FIG 1 (Summer/Winter NBP)
# ==========================================
print("Fetching Fig 1 data...")
current_date = datetime.date.today()
ed = current_date.strftime('%Y-%m-%d')
sd = (current_date - timedelta(days=365)).strftime('%Y-%m-%d')

fields = ['volume', 'close']
my_data = icexl.get_timeseries(['GWMS 1!-ICE'], fields, granularity='D', start_date=sd, end_date=ed)
my_data2 = icexl.get_timeseries(['GWMS 2!-ICE'], fields, granularity='D', start_date=sd, end_date=ed)

dates, vols, values = zip(*my_data[1:])  
dates2, vols2, values2 = zip(*my_data2[1:])  
formatted_dates = [date for date in dates]

num_points = len(formatted_dates)
indices = [0, num_points//3, 2*num_points//3, num_points-1]
selected_dates = [formatted_dates[i] for i in indices]

fig1 = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.1, row_heights=[0.99, 0.01])
fig1.add_trace(go.Scatter(x=formatted_dates, y=values, mode='lines+markers',marker=dict(color='deepskyblue'), name='Summer 26'),row=1, col=1)
fig1.add_trace(go.Scatter(x=formatted_dates, y=values2, mode='lines+markers',marker=dict(color='darkviolet'), name='Winter 26'),row=1, col=1)

fig1.update_layout(
    title={'text': 'Summer and Winter 26 Daily NBP Prices 12 Month Lag', 'x': 0.5, 'xanchor': 'center'},
    legend=dict(x=0.01, y=0.98, bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1),
    yaxis_title='p/therm',
    xaxis=dict(tickmode='array', tickvals=[formatted_dates[i] for i in indices], ticktext=selected_dates),
    font=dict(size=24), 
    autosize=True, margin=dict(l=50, r=30, t=60, b=50)
)

# ==========================================
# 2. FETCH DATA & CREATE FIG 2 (Forward NBP WoW)
# ==========================================
print("Fetching Fig 2 data...")
sd_7 = (current_date - timedelta(days=7)).strftime('%Y-%m-%d')
lom = ['GWM ' + str(i) + '!-ICE' for i in range(1,13)]
ed = (current_date - timedelta(days=1)).strftime('%Y-%m-%d')
sd = (current_date - timedelta(days=8)).strftime('%Y-%m-%d')

my_data_wow = icexl.get_timeseries(lom, ['Last'], granularity='D', start_date=sd, end_date=ed)

last_week_p = my_data_wow[1][1:]

wow_pull = icexl.get_quotes(lom, ['ICE Theoretical Price'])
current_theo_p = [i[1] for i in wow_pull[1:]]

def get_next_12_months():
    start_d = datetime.date.today() + datetime.timedelta(days=1)
    months_list = []
    curr_d = start_d
    for _ in range(12):
        curr_m = curr_d.month
        curr_y = curr_d.year
        next_m = curr_m + 1
        next_y = curr_y
        if next_m > 12:
            next_m = 1
            next_y += 1
        curr_d = datetime.date(next_y, next_m, 1)
        months_list.append(curr_d.strftime('%b'))
    return months_list

labels = get_next_12_months()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=labels, y=last_week_p, mode='lines+markers', name='Last Week', marker=dict(color='deepskyblue'), line=dict(color='deepskyblue')))
fig2.add_trace(go.Scatter(x=labels, y=current_theo_p, mode='lines+markers', name='Current Price', marker=dict(color='darkviolet'), line=dict(color='darkviolet')))

fig2.update_layout(
    title={'text': 'Forward NBP Curve Change Week on Week', 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Month', yaxis_title='P/th',
    legend=dict(x=0.01, y=0.98, bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1),
    hovermode='x unified',
    font=dict(size=24), autosize=True, margin=dict(l=50, r=30, t=60, b=50)
)

# ==========================================
# 3. SET UP MARINE TRAFFIC IFRAME
# ==========================================
print("Generating MarineTraffic Iframe...")
# We use the /embed/ endpoint to bypass X-Frame-Options blocking
marine_traffic_iframe = """
<iframe 
    src="https://www.marinetraffic.com/en/ais/embed/zoom:8/centery:26.2/centerx:56.5/maptype:1" 
    width="100%" 
    height="100%" 
    style="min-height: 400px; border: none; border-radius: 8px;" 
    frameborder="0" 
    scrolling="no">
</iframe>
"""

# ==========================================
# 4. FETCH DATA & CREATE FIG 4 (GAS Forward Curve WoW)
# ==========================================
print("Fetching Fig 4 data (GAS Curve)...")
ed_gas = (current_date - timedelta(days=0)).strftime('%Y-%m-%d')
sd_gas = (current_date - timedelta(days=7)).strftime('%Y-%m-%d')
lom_gas = ['GAS ' + str(i) + '!-ICE' for i in range(1,13)]

my_data_gas = icexl.get_timeseries(lom_gas, ['Last'], granularity='D', start_date=sd_gas, end_date=sd_gas)
last_week_gas_p = my_data_gas[1][1:]

wow_pull_gas = icexl.get_quotes(lom_gas, ['ICE Theoretical Price'])
current_theo_gas_p = [i[1] for i in wow_pull_gas[1:]]

fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=labels, # Reusing labels array initialized in Fig 2
    y=last_week_gas_p, 
    mode='lines+markers', 
    name='Last Week',
    marker=dict(color='deepskyblue'),
    line=dict(color='deepskyblue')
))
fig4.add_trace(go.Scatter(
    x=labels, 
    y=current_theo_gas_p, 
    mode='lines+markers', 
    name='Current Price',
    marker=dict(color='darkviolet'),
    line=dict(color='darkviolet')
))

fig4.update_layout(
    title={'text': 'Forward Diesel/Heating Oil Curve Change Week on Week', 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Month', yaxis_title='USD ($) /MT',
    legend=dict(x=0.01, y=0.98, bgcolor='rgba(255, 255, 255, 0.8)', bordercolor='lightgray', borderwidth=1),
    hovermode='x unified',
    font=dict(size=24), autosize=True, margin=dict(l=50, r=30, t=60, b=50)
)

# ==========================================
# 5. FETCH NEWS (EventRegistry)
# ==========================================
print("Fetching News...")
er = EventRegistry(apiKey = '1312ee84-a9f9-4fc6-95a4-0ed13b135ec3')
nat_gas_concept_uri = er.getConceptUri("Natural Gas")
q = QueryArticles(conceptUri=nat_gas_concept_uri, lang="eng")
q.setRequestedResult(RequestArticlesInfo(sortBy="date", count=100))
results = er.execQuery(q)

unique_list = []
seen_titles = set()
if results.get('articles') and results['articles'].get('results'):
    for article in results['articles']['results']:
        title = article['title'].strip()
        source_name = article['source']['title']
        if re.search('gas', title, re.IGNORECASE) and title not in seen_titles:
            unique_list.append(f"{title} - {source_name}")
            seen_titles.add(title)

display_news = unique_list + unique_list 

# ==========================================
# 6. GENERATE HTML DASHBOARD
# ==========================================
print("Generating Final HTML Dashboard...")

fig1_html = fig1.to_html(full_html=False, include_plotlyjs='cdn')
fig2_html = fig2.to_html(full_html=False, include_plotlyjs=False)
fig3_html = marine_traffic_iframe # Pushing the embedded map to the Jinja template
fig4_html = fig4.to_html(full_html=False, include_plotlyjs=False)

env = Environment(loader=FileSystemLoader('.'))
template = env.get_template('tv_template.html')

html_content = template.render(
    fig1_html=fig1_html,
    fig2_html=fig2_html,
    fig3_html=fig3_html,
    fig4_html=fig4_html,
    headlines=display_news
)

output_file = 'index.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"Success! {output_file} has been generated.")

Fetching Fig 1 data...
Fetching Fig 2 data...
Generating MarineTraffic Iframe...
Fetching Fig 4 data (GAS Curve)...
Fetching News...
Generating Final HTML Dashboard...
Success! index.html has been generated.
